# Dotplots

In [ ]:
import scanpy as sc
import infercnvpy as cnv
import matplotlib.pyplot as plt
import warnings
import pandas as pd
import seaborn as sns
import muon as mu

warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['figure.figsize'] = (5, 5)
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
adata_rna = sc.read_h5ad("13052025_updated_annotations_colors_scores.h5ad")
adata_rna.obs.index = adata_rna.obs["dataset_name"].astype('str') + "_" + adata_rna.obs.index
adata_rna.obs.index

In [ ]:
adata_protein = sc.read_h5ad("/home/prisb/projects/henry_hashtag_experiment/exmds/combined_protein.h5ad")
adata_protein.obs.index = adata_protein.obs["dataset"].astype('str') + "_" + adata_protein.obs.index
adata_protein.obs

In [ ]:
adata_rna.obs_names.duplicated().sum()

In [ ]:
adata_protein.obs_names.duplicated().sum()

## Merge the rna and protein

In [ ]:
mdata = mu.MuData({"rna":adata_rna, "protein":adata_protein})
mdata

In [ ]:
# check that everything is in order
mu.pl.embedding(mdata, basis="rna:X_umap", color=["rna:celltype_v2"], legend_loc="on data", legend_fontsize=5)

Dotplots for annotations 'healthy cells'. Email from Henry

For the gene expression dot plot could you add the following cells?
CD34
MKI67
PCNA

For the gene expression plot could you remove the following molecules?
THY1
CLEC9A
CR1
ALDH1A1
HOXB4SELP
VWF
ENG
ANPEP
FCGR1A
 
For the antibody expression plot could you remove the following molecules?
CD370
CD35
CD62P
CD105 
CD13
CD64

In [ ]:
markersfile = pd.read_csv("data/2025_04_04_Molecules_for_Priscilla(for_dotplots).csv")

In [ ]:
markersfile = markersfile.fillna("na")
markersfile

In [ ]:
new_adtmarkersfile = pd.read_csv("data/2025_05_02_ADT_Markers_Atypical_for_Priscilla.csv")
new_adtmarkersfile

In [ ]:
new_adtmarkersfile["ADT"] =str("ADT.")+ new_adtmarkersfile["Marker"]
new_adtmarkersfile

In [ ]:
new_adtmarkersfile["ADT"].tolist()

In [ ]:
mdata['protein'].obs['celltype_v2'] = mdata['rna'].obs['celltype_v2']

In [ ]:
mdata['protein'].obs['celltype_v2'].unique()

In [ ]:
healthy_ct = ["HSC", 'MPP', 'MEP/MKP', 'MEP', 'Erythroblast', 'BMCP','GMP','Monocyte progenitor']

In [ ]:
#inspect the expression value of the ADT controls to determine a cutoff of "expression"

isotypes = mdata['protein'].var_names[mdata['protein'].var_names.str.contains("Ctrl")].tolist()

fig, ax = plt.subplots(1,1,figsize=(15,5))
sc.pl.violin(mdata['protein'], isotypes+new_adtmarkersfile["ADT"].tolist(), stripplot=False, rotation=90, ax=ax)

fig1, ax1 = plt.subplots(1,1,figsize=(15,5))
sc.pl.violin(mdata['protein'], isotypes+new_adtmarkersfile["ADT"].tolist(), stripplot=False, rotation=90, layer="counts", ax=ax1)
#sc.pl.violin(mdata['protein'], isotypes, stripplot=False, rotation=90)

In [ ]:
sub = mdata['protein'][:, isotypes]
sub.X.mean(axis=0).mean()

### Atypical ADT markers dotplot

In [ ]:
atypical_adtlist = [
 'ADT.CD109',
 'ADT.CD194',
 'ADT.CD324',
 'ADT.CD326',
 'ADT.CD35',
 'ADT.CD36',
 'ADT.CD56',
 'ADT.CD45RA',
 'ADT.integrin',
 'ADT.CD49b',
 'ADT.CD371',
 'ADT.GPR56',
 'ADT.CD164',
 'ADT.CD172a',
 'ADT.CD4',
 'ADT.CD9',
 'ADT.CD25',
 'ADT.CD62L',
 'ADT.KLRG1',
 'ADT.CD86',
 'ADT.CD101']


In [ ]:
# Dotplot for ADT expression in Atypical cells

fig, ax = plt.subplots(1,1,figsize=(15,10))

sc.pl.dotplot(mdata['protein'][~mdata['protein'].obs['celltype_v2'].isin(healthy_ct)], 
    atypical_adtlist, 
    groupby="celltype_v2", 
    #categories_order=healthy_ct, 
    dendrogram=False,
    expression_cutoff=2.35, 
    ax=ax,
    show=False,
    use_raw=False)

#fig.savefig("figures/ADT_atypical_dotplot_v4.pdf", bbox_inches='tight')


In [ ]:
atypical_adt_cellorder = [
#
"HSC", 'MPP', 'MEP/MKP', 'MEP', 'Erythroblast', 'BMCP','GMP','Monocyte progenitor'   
]

In [ ]:
# Dotplot for ADT expression in Atypical cells with the healthy cells displayed

fig, ax = plt.subplots(1,1,figsize=(15,10))

mdata['protein'].obs['celltype_v2'] = mdata['protein'].obs['celltype_v2'].cat.remove_unused_categories()

sc.pl.matrixplot(mdata['protein'], 
    ["ADT.CD34"]+atypical_adtlist, 
    groupby="celltype_v2", 
    categories_order=atypical_adt_cellorder, 
    dendrogram=True, 
    ax=ax,
    show=False, 
    cmap="Reds",
    )

#fig.savefig("figures/ADT_atypical_healthy_matrixplot_v1.pdf", bbox_inches='tight')


### ADT markers healthy dotplot

In [ ]:
#remove ADT.CD370, ADT.CD35, ADT.CD62P, ADT.CD105, ADT.CD13, ADT.CD64

remove_adt = ["ADT.CD370", "ADT.CD35", "ADT.CD62P", "ADT.CD105", "ADT.CD13", "ADT.CD64","na"]

healthy_adt = markersfile[~markersfile["Antibody"].isin(remove_adt)]['Antibody'].tolist()
healthy_adt = ["ADT.CD34"] + healthy_adt
healthy_adt


In [ ]:
# Dotplot for ADT expression in healthy cells

fig, ax = plt.subplots(1,1,figsize=(15,5))

sc.pl.dotplot(mdata['protein'][mdata['protein'].obs['celltype_v2'].isin(healthy_ct)], 
    healthy_adt, 
    groupby="celltype_v2", 
    categories_order=healthy_ct, 
    dendrogram=True,
    expression_cutoff=2.35, 
    ax=ax,
    show=False,
    vmax=17.5)

fig.savefig("figures/ADT_healthy_dotplot_v4.pdf", bbox_inches='tight')

### Gene expression dot plot for healthy cells

In [ ]:
#manually edited list and positions
healthy_genes = ["PROM1", "ADGRG1", "CD34", "CD38", "PCNA", "GATA1", "TFRC", "CD36", "ENPP3", "FCER1A", "MPO", "CD33", "CD48", "CD14"]

In [ ]:
#visualise the healthy_genes list and means and create a dataframe

adata_rna[:, "MPO"].X.max()

In [ ]:
sc.pl.violin(adata_rna, healthy_genes, rotation=90, use_raw=False)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,5))
sc.pl.dotplot(mdata['rna'][mdata['rna'].obs['celltype_v2'].isin(healthy_ct)], 
    healthy_genes,
    groupby="celltype_v2", 
    categories_order=healthy_ct, 
    dendrogram=False,
    ax=ax, 
    show=False,
    use_raw=False)

fig.savefig("figures/GEX_healthy_dotplot_v5.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10,10))
sc.pl.dotplot(mdata['rna'][mdata['rna'].obs['celltype_v2'].isin(atypical_adt_cellorder)], 
    healthy_genes,
    groupby="celltype_v2", 
    dendrogram=True,
    ax=ax, 
    show=False,
    use_raw=False,
    standard_scale="var")

fig.savefig("figures/GEX_healthy_atypical_dotplot_v1.pdf", bbox_inches='tight')

In [ ]:
#mdata.write_h5mu("06052025_gex_adt_celltypev2_updated_colors.h5mu")

In [ ]:
sc.pl.umap(adata_rna, color=["MPO"])

In [ ]:
sc.pl.violin(mdata['rna'], 'MPO', rotation=90, groupby='celltype_v2', use_raw=False)

## MISC

In [ ]:
mu.pl.embedding(mdata[mdata.obs['rna:patient_alias'] == "P01"], basis="protein:X_umap", 
color=["rna:chr17_76736877_G_T_call","rna:chr17_76736877_G_C_call","rna:chr17_76736877_G_A_call"],
groups=["mt",'wt'])

In [ ]:
mu.pl.embedding(mdata[mdata.obs['rna:patient_alias'] == "P02"], basis="protein:X_umap", 
color=["rna:chr17_76736877_G_T_call","rna:chr17_76736877_G_C_call","rna:chr17_76736877_G_A_call"],
groups=["mt",'wt'])

In [ ]:
mu.pl.embedding(mdata[mdata.obs['rna:patient_alias'] == "P08"], basis="protein:X_umap", 
color=["rna:chr17_76736877_G_T_call","rna:chr17_76736877_G_C_call","rna:chr17_76736877_G_A_call"],
groups=["mt",'wt'])

In [ ]:
sc.tl.pca(mdata.mod['protein'])
sc.pp.neighbors(mdata.mod['protein'])
sc.tl.umap(mdata.mod['protein'])
sc.pl.umap(mdata.mod['protein'])

In [ ]:
sc.tl.leiden(mdata.mod['protein'])

In [ ]:
mdata.update()

In [ ]:
mu.pl.embedding(mdata, basis="protein:X_umap", color=["protein:celltype_v2"], groups=["Atypical cluster J","Atypical cluster N"])

In [ ]:
mu.pl.embedding(mdata, basis="protein:X_umap", color=["ADT.CD9","ADT.CD371",'ADT.CD62L'])

In [ ]:
sc.pl.umap(adata_rna, color='ASXL1')

In [ ]:
sc.tl.rank_genes_groups(mdata.mod['protein'], groupby='leiden', groups=["17","25"],n_genes=20, method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(mdata.mod['protein'])

In [ ]:
mu.pl.embedding(mdata, basis="protein:X_umap", color=["ADT.B2M","ADT.CD36"])

In [ ]:
mu.pl.embedding(mdata, basis="protein:X_umap", color=["rna:new_celltype","rna:patient_alias"], ncols=1)

In [ ]:
mdata

In [ ]:
mu.pl.embedding(mdata[mdata.obs['rna:patient_alias'] == "P18"], basis="protein:X_umap", color=["rna:infercnv_new_celltype","rna:timepoint_coarse"])

In [ ]:
atypical_adt_cellorder2 = ['Atypical cluster A',
 'Atypical cluster B',
 'Atypical cluster C',
 'Atypical cluster D',
 'Atypical cluster E',
 'Atypical cluster G',
 'Atypical cluster H',
 'Atypical cluster I',
 'Atypical cluster J',
 'Atypical cluster K',
 'Atypical cluster L',
 'Atypical cluster M',
 'Atypical cluster N',
 'HSC',
 'MPP',
 'MEP/MKP',
 'MEP',
 'Erythroblast',
 'BMCP',
 'GMP',
 'Monocyte progenitor']

In [ ]:
sc.tl.rank_genes_groups(adata_protein, groupby='celltype_v2', groups=atypical_adt_cellorder2, n_genes=20, method='wilcoxon') #adata_protein

In [ ]:
adata_protein.uns['rank_genes_groups']

In [ ]:
sc.pl.rank_genes_groups(adata_protein)

# dendrograms based on transcriptome and antibody markers

In [ ]:
adata_rna

In [ ]:
sc.tl.dendrogram(adata_rna, groupby='celltype_v2', use_raw=False, use_rep="X_pca")

In [ ]:
adata_rna_dendrogram_order = adata_rna.uns['dendrogram_celltype_v2']['categories_ordered']
adata_rna_dendrogram_order

In [ ]:
adata_rna_dendrogram_order_healthy = ['Erythroblast',
 'MEP',
 'MEP/MKP',
 'HSC',
 'MPP',
 'Monocyte progenitor',
 'GMP',
 'BMCP',]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,5))
sc.pl.dendrogram(adata_rna, groupby='celltype_v2', dendrogram_key="dendrogram_celltype_v2", orientation="left", ax=ax, show=False)
fig.savefig("figures/gex_dendrogram.pdf", dpi=200, bbox_inches='tight')

In [ ]:
#gex dotplot based on the total hierachical clustering dendrogram

fig, ax = plt.subplots(1,1,figsize=(10,10))
sc.pl.dotplot(adata_rna, 
    healthy_genes,
    categories_order=adata_rna_dendrogram_order,
    groupby="celltype_v2", 
    dendrogram=False,
    ax=ax, 
    show=False,
    use_raw=False,
    standard_scale="var")

fig.savefig("figures/GEX_healthy_atypical_dotplot_v2.pdf", bbox_inches='tight')

In [ ]:
#gex healthy only dotplot based on the total hierachical clustering dendrogram

fig, ax = plt.subplots(1,1,figsize=(6,4))
sc.pl.dotplot(adata_rna[adata_rna.obs['celltype_v2'].isin(healthy_ct)], 
    healthy_genes,
    categories_order=adata_rna_dendrogram_order_healthy,
    groupby="celltype_v2", 
    dendrogram=False,
    ax=ax, 
    show=False,
    use_raw=False,
    standard_scale="var")

fig.savefig("figures/GEX_healthy_dotplot_v6.pdf", bbox_inches='tight')

In [ ]:
mdata['protein'].obs['celltype_v2'] = mdata['rna'].obs['celltype_v2']

In [ ]:
mdata

In [ ]:
prot = mdata.mod['protein'].copy()
prot.obs["celltype_v2"] = prot.obs["celltype_v2"].cat.remove_unused_categories()

In [ ]:
sc.tl.dendrogram(prot, groupby="celltype_v2", use_raw=False, use_rep="X_pca")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,5))
sc.pl.dendrogram(prot, groupby="celltype_v2", dendrogram_key="dendrogram_celltype_v2", orientation="left", ax=ax, show=False)
ax.set_title("Dendrogram based on antibody markers")
fig.savefig("figures/adt_dendrogram.pdf", dpi=200, bbox_inches='tight')

# Differential gene and antibody expression

In [ ]:
prot = mdata.mod['protein'].copy()
prot

In [ ]:
healthy_ct

In [ ]:
exclude_ct=['Atypical cluster O','Atypical cluster F']
prot2= prot[~prot.obs['celltype_v2'].isin(exclude_ct)].copy()
prot2.obs['classification'] = prot2.obs['celltype_v2'].apply(lambda ct: 'Healthy' if ct in healthy_ct else ct)
prot2.obs['classification'] = prot2.obs['classification'].fillna("na")
prot2.obs['classification']


In [ ]:
prot2.obs['classification'].value_counts()

In [ ]:
atypical_adt_cellorder3 = [
'Atypical cluster A', 'Atypical cluster B', 'Atypical cluster C',
'Atypical cluster D', 'Atypical cluster E',
'Atypical cluster G', 'Atypical cluster H', 'Atypical cluster I',
'Atypical cluster J', 'Atypical cluster K', 'Atypical cluster L',
'Atypical cluster M', 'Atypical cluster N'  ]

In [ ]:
sc.tl.rank_genes_groups(prot2, groupby='classification', groups=atypical_adt_cellorder3, reference ='Healthy', method='wilcoxon') #adata_protein

In [ ]:
sc.pl.rank_genes_groups(prot2)

In [ ]:
sc.pl.rank_genes_groups_dotplot(prot2, min_logfoldchange=2.0)

In [ ]:
prot2.uns['rank_genes_groups']

In [ ]:
prot2_df = sc.get.rank_genes_groups_df(prot2, group=None)

In [ ]:
prot2_df.to_csv("ADT_atypical_clusters_rank_genes_groups_df.csv")

In [ ]:
adata_rna.obs['celltype_v3'] = adata_rna.obs['celltype_v2'].copy()
adata_rna.obs['celltype_v3'] = adata_rna.obs['celltype_v3'].cat.add_categories(['HSC_P09'])
adata_rna.obs.loc[
    (adata_rna.obs['patient_alias'] == 'P09') & (adata_rna.obs['celltype_v3'] == 'HSC'),
    'celltype_v3'
] = 'HSC_P09'
adata_rna.obs[['patient_alias', 'celltype_v3']]

In [ ]:
mdata['rna'].obs['celltype_v3'] = adata_rna.obs['celltype_v3'].copy()
mdata['rna'].obs[['celltype_v2','celltype_v3']]

In [ ]:
#checking that both adata_rna and mdata['rna'] have the same index
mdata['rna'].obs.index.equals(adata_rna.obs.index)

In [ ]:
mdata['protein'].obs['celltype_v3'] = mdata['rna'].obs['celltype_v3'].copy()
mdata

In [ ]:
mdata['protein'].obs['timepoint'] = mdata['rna'].obs['timepoint'].copy()
mdata.update()
mdata.obs

In [ ]:
rna1 = mdata.mod['rna'][mdata.mod['rna'].obs['celltype_v3'].isin(healthy_ct+['HSC_P09'])]
rna1.obs['celltype_v3'].unique()

In [ ]:
import numpy as np
rna1.obs['classification'] = np.where(rna1.obs['celltype_v3'].isin(healthy_ct), 'healthy', 'HSC_P09')
rna1.obs[['celltype_v3', 'classification']]

In [ ]:
prot1 = mdata.mod['protein'][mdata.mod['protein'].obs['celltype_v3'].isin(healthy_ct+['HSC_P09'])]
prot1

In [ ]:
prot1.obs['classification'] = np.where(prot1.obs['celltype_v3'].isin(healthy_ct), 'healthy', 'HSC_P09')
prot1.obs[['celltype_v3', 'classification']]

In [ ]:
# HSC_P09 vs HSC in rest in GEX

sc.tl.rank_genes_groups(rna1, 
    groupby='celltype_v3', 
    groups=['HSC', 'HSC_P09'], 
    reference='HSC', 
    method='wilcoxon',
    key_added="HSCP09_vs_HSC",
    use_raw=False)


In [ ]:
sc.get.rank_genes_groups_df(rna1, key="HSCP09_vs_HSC", group="HSC_P09").to_csv("data/HSC_P09_vs_HSC_rank_GEX_df.csv")

In [ ]:
# HSC_P09 vs HSC in rest in ADT

sc.tl.rank_genes_groups(prot1, 
    groupby='celltype_v3', 
    groups=['HSC', 'HSC_P09'], 
    reference='HSC', 
    method='wilcoxon',
    key_added="HSCP09_vs_HSC",
    use_raw=False)

In [ ]:
sc.get.rank_genes_groups_df(prot1, key="HSCP09_vs_HSC", group="HSC_P09").to_csv("data/HSC_P09_vs_HSC_rank_ADT_df.csv")

In [ ]:
# HSC_P09 vs other healthy celltypes in GEX

sc.tl.rank_genes_groups(rna1, 
    groupby='classification', 
    groups=['HSC_P09','healthy'], 
    reference='healthy', 
    method='wilcoxon',
    key_added="HSCP09_vs_healthy",
    use_raw=False)

In [ ]:
sc.get.rank_genes_groups_df(rna1, key="HSCP09_vs_healthy", group="HSC_P09").to_csv("data/HSC_P09_vs_healthy_rank_GEX_df.csv")

In [ ]:
# HSC_P09 vs other healthy celltypes in ADT

sc.tl.rank_genes_groups(prot1, 
    groupby='classification', 
    groups=['HSC_P09','healthy'], 
    reference='healthy', 
    method='wilcoxon',
    key_added="HSCP09_vs_healthy",
    use_raw=False)

sc.get.rank_genes_groups_df(prot1, key="HSCP09_vs_healthy", group="HSC_P09").to_csv("data/HSC_P09_vs_healthy_rank_ADT_df.csv")

In [ ]:
temprna1 = rna1[((rna1.obs['timepoint'] == "C1D1") & (rna1.obs['celltype_v3'] == "HSC_P09")) | (rna1.obs['celltype_v3'] == "HSC")].copy()

sc.tl.rank_genes_groups(temprna1,
    groupby='celltype_v3',
    groups=['HSC', 'HSC_P09'],
    reference='HSC',
    method='wilcoxon',
    key_added="HSC_vs_HSC_P09_C1D1",
    use_raw=False)

sc.get.rank_genes_groups_df(temprna1, key="HSC_vs_HSC_P09_C1D1", group="HSC_P09").to_csv("data/HSC_vs_HSC_P09_C1D1_rank_GEX_df.csv")

In [ ]:
tempprot1 = mdata['protein'][((mdata['protein'].obs['timepoint'] == "C1D1") & (mdata['protein'].obs['celltype_v3'] == "HSC_P09")) | (mdata['protein'].obs['celltype_v3'] == "HSC")].copy()

sc.tl.rank_genes_groups(tempprot1,
    groupby='celltype_v3',
    groups=['HSC', 'HSC_P09'],
    reference='HSC',
    method='wilcoxon',
    key_added="HSC_vs_HSC_P09_C1D1",
    use_raw=False)

sc.get.rank_genes_groups_df(tempprot1, key="HSC_vs_HSC_P09_C1D1", group="HSC_P09").to_csv("data/HSC_vs_HSC_P09_C1D1_rank_ADT_df.csv")

In [ ]:
temprna2 = rna1[((rna1.obs['timepoint'] == "C1D1") & (rna1.obs['celltype_v3'] == "HSC_P09")) | (rna1.obs['classification'] == "healthy")].copy()

sc.tl.rank_genes_groups(temprna2,
    groupby='classification',
    groups=['healthy', 'HSC_P09'],
    reference='healthy',
    method='wilcoxon',
    key_added="HSC_P09_C1D1_vs_healthy",
    use_raw=False)

sc.get.rank_genes_groups_df(temprna2, key="HSC_P09_C1D1_vs_healthy", group="HSC_P09").to_csv("data/HSC_P09_C1D1_vs_healthy_rank_GEX_df.csv")

In [ ]:
tempprot2 = mdata['protein'][mdata.mod['protein'].obs['celltype_v3'].isin(healthy_ct+['HSC_P09'])]
tempprot2.obs['classification'] = np.where(tempprot2.obs['celltype_v3'].isin(healthy_ct), 'healthy', 'HSC_P09')
tempprot2 = tempprot2[((tempprot2.obs['timepoint'] == "C1D1") & (tempprot2.obs['celltype_v3'] == "HSC_P09")) | (tempprot2.obs['classification'] == "healthy")]

sc.tl.rank_genes_groups(tempprot2,
    groupby='classification',
    groups=['healthy', 'HSC_P09'],
    reference='healthy',
    method='wilcoxon',
    key_added="HSC_P09_C1D1_vs_healthy",
    use_raw=False)

sc.get.rank_genes_groups_df(tempprot2, key="HSC_P09_C1D1_vs_healthy", group="HSC_P09").to_csv("data/HSC_P09_C1D1_vs_healthy_rank_ADT_df.csv")

In [ ]:
temprna3 = rna1[((rna1.obs['timepoint'] == "C7D1") & (rna1.obs['celltype_v3'] == "HSC_P09")) | (rna1.obs['celltype_v3'] == "HSC")].copy()

sc.tl.rank_genes_groups(temprna3,
    groupby='celltype_v3',
    groups=['HSC', 'HSC_P09'],
    reference='HSC',
    method='wilcoxon',
    key_added="HSC_vs_HSC_P09_C7D1",
    use_raw=False)

sc.get.rank_genes_groups_df(temprna3, key="HSC_vs_HSC_P09_C7D1", group="HSC_P09").to_csv("data/HSC_vs_HSC_P09_C7D1_rank_GEX_df.csv")

In [ ]:
temprna3.obs[['celltype_v3','timepoint']].value_counts()

In [ ]:
tempprot3 = mdata['protein'][((mdata['protein'].obs['timepoint'] == "C7D1") & (mdata['protein'].obs['celltype_v3'] == "HSC_P09")) | (mdata['protein'].obs['celltype_v3'] == "HSC")].copy()

sc.tl.rank_genes_groups(tempprot3,
    groupby='celltype_v3',
    groups=['HSC', 'HSC_P09'],
    reference='HSC',
    method='wilcoxon',
    key_added="HSC_vs_HSC_P09_C7D1",
    use_raw=False)

sc.get.rank_genes_groups_df(tempprot3, key="HSC_vs_HSC_P09_C7D1", group="HSC_P09").to_csv("data/HSC_vs_HSC_P09_C7D1_rank_ADT_df.csv")

In [ ]:
tempprot3.obs[['celltype_v3','timepoint']].value_counts()

In [ ]:
temprna4 = rna1[((rna1.obs['timepoint'] == "C7D1") & (rna1.obs['celltype_v3'] == "HSC_P09")) | (rna1.obs['classification'] == "healthy")].copy()

sc.tl.rank_genes_groups(temprna4,
    groupby='classification',
    groups=['healthy', 'HSC_P09'],
    reference='healthy',
    method='wilcoxon',
    key_added="HSC_P09_C7D1_vs_healthy",
    use_raw=False)

sc.get.rank_genes_groups_df(temprna4, key="HSC_P09_C7D1_vs_healthy", group="HSC_P09").to_csv("data/HSC_P09_C7D1_vs_healthy_rank_GEX_df.csv")

In [ ]:
temprna4.obs[['classification','timepoint']].value_counts()

In [ ]:
tempprot4 = mdata['protein'][mdata.mod['protein'].obs['celltype_v3'].isin(healthy_ct+['HSC_P09'])]
tempprot4.obs['classification'] = np.where(tempprot4.obs['celltype_v3'].isin(healthy_ct), 'healthy', 'HSC_P09')
tempprot4 = tempprot4[((tempprot4.obs['timepoint'] == "C7D1") & (tempprot4.obs['celltype_v3'] == "HSC_P09")) | (tempprot4.obs['classification'] == "healthy")]

sc.tl.rank_genes_groups(tempprot4,
    groupby='classification',
    groups=['healthy', 'HSC_P09'],
    reference='healthy',
    method='wilcoxon',
    key_added="HSC_P09_C7D1_vs_healthy",
    use_raw=False)

sc.get.rank_genes_groups_df(tempprot4, key="HSC_P09_C7D1_vs_healthy", group="HSC_P09").to_csv("data/HSC_P09_C7D1_vs_healthy_rank_ADT_df.csv")

In [ ]:
tempprot4.obs[['classification','timepoint']].value_counts()